# RAG Pipeline: Question Answering over PDFs

This notebook builds a small, beginner-friendly **Retrieval-Augmented Generation (RAG)** system
from scratch, step by step:

**PDFs → Documents → Chunks → Embeddings → Vector Store (ChromaDB) → Retrieval → LLM Answer**

RAG works in two halves:
- **Retrieval half**: find the pieces of text in our PDFs that are most relevant to a question.
- **Generation half**: hand those pieces to an LLM as "context" so it can answer using real
  information instead of guessing from memory.

Libraries used: `langchain` (document loading/splitting), `sentence-transformers` (embeddings),
`chromadb` (vector database), `langchain-groq` (fast, free LLM inference via Groq).

## 0. Setup

First we create a folder where PDFs will live, and install the libraries we need.

In [1]:
import os

# All source PDFs will be dropped into this folder before we run the ingestion pipeline.
PDF_FOLDER = "data/pdfs"
os.makedirs(PDF_FOLDER, exist_ok=True)

print("PDF folder ready at:", PDF_FOLDER)

PDF folder ready at: data/pdfs


In [2]:
# Sanity check — see what PDFs are currently sitting in the folder.
# (Upload your PDF(s) into data/pdfs before running this.)
os.listdir(PDF_FOLDER)

[]

In [7]:
# Install every package the pipeline depends on.
#   langchain / langchain-community -> PDF loading + text splitting utilities
#   langchain-groq                  -> chat interface for Groq-hosted LLMs
#   pypdf / pymupdf                 -> PDF parsing engines used under the hood
#   sentence-transformers           -> turns text into embedding vectors
#   chromadb                        -> local vector database

!pip install langchain langchain-core langchain-community langchain-groq pypdf pymupdf sentence-transformers chromadb -q

In [8]:
from langchain_core.documents import Document

# `Document` is LangChain's standard container: some text (page_content) plus
# a metadata dictionary describing where that text came from. Every loader,
# splitter, and retriever in this notebook passes Document objects around.
demo_document = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)
demo_document

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

## Ingestion Pipeline

This half of the notebook turns raw PDFs into searchable vectors:

**Raw PDFs → Document objects → smaller chunks → embeddings → saved in ChromaDB**

### Step 1 — Load the PDFs

In [9]:
from langchain_community.document_loaders.pdf import PyPDFLoader


def load_all_pdfs(folder_path=PDF_FOLDER):
    """Load every PDF inside `folder_path` and return one Document per page.

    Each PDF is opened with PyPDFLoader, which splits it into a list of
    Document objects (one per page). We simply collect all of those
    page-level Documents from every PDF into a single flat list.
    """
    loaded_documents = []
    pdf_count = 0

    for filename in os.listdir(folder_path):
        if not filename.lower().endswith(".pdf"):
            continue  # skip anything that isn't a PDF

        pdf_path = os.path.join(folder_path, filename)
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()  # -> list[Document], one entry per page

        loaded_documents.extend(pages)
        pdf_count += 1

    print("PDFs loaded:", pdf_count)
    print("Total pages loaded:", len(loaded_documents))
    return loaded_documents

In [12]:
raw_documents = load_all_pdfs()

# Safety check: if this is 0, either data/pdfs is empty, or you uploaded the
# PDF AFTER already running this cell once. Re-run this cell after uploading.
assert len(raw_documents) > 0, (
    "No pages loaded! Upload a PDF into the data/pdfs folder, then re-run "
    "THIS cell (and every cell below it) again, in order."
)

PDFs loaded: 1
Total pages loaded: 21


### Step 2 — Split into chunks

Whole pages are usually too big and too noisy to embed well: a page can mix multiple
topics, which blurs the meaning of its embedding. Splitting into smaller, overlapping
chunks keeps each vector focused on one idea, which makes retrieval far more precise.

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_into_chunks(documents, chunk_size=500, chunk_overlap=50):
    """Break large Documents into smaller overlapping chunks.

    chunk_size    -> max characters per chunk
    chunk_overlap -> characters shared between consecutive chunks, so a
                      sentence that gets cut in half still has context on
                      both sides
    """
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    return splitter.split_documents(documents)

In [14]:
document_chunks = split_into_chunks(raw_documents)
print("Total chunks created:", len(document_chunks))

assert len(document_chunks) > 0, (
    "No chunks created! Go back and re-run the 'raw_documents = load_all_pdfs()' "
    "cell first, then re-run this cell."
)

Total chunks created: 244


### Step 3 — Generate embeddings

An embedding is just a list of numbers (a vector) that represents the *meaning* of a
piece of text. Texts with similar meaning end up with vectors that are close together,
which is exactly what lets us do semantic search later.

In [15]:
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Loads a SentenceTransformer model and turns text into embedding vectors."""

    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading embedding model:", self.model_name)

        self.model = SentenceTransformer(self.model_name)
        print("Embedding dimension:", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, texts):
        """Encode a list of strings into a NumPy array of embedding vectors."""
        vectors = self.model.encode(texts, show_progress_bar=True)
        print("Embeddings shape:", vectors.shape)
        return vectors

In [16]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dimension: 384


/tmp/ipykernel_701/3998722253.py:12: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", self.model.get_sentence_embedding_dimension())


### Step 4 — Store embeddings in ChromaDB

ChromaDB is a lightweight vector database that saves each chunk's text, metadata, and embedding to disk, and lets us later ask "which stored vectors are closest to this query vector?"

In [17]:
import uuid
import chromadb


class VectorStoreManager:
    """A thin wrapper around a persistent ChromaDB collection.

    Handles creating/opening the collection on disk and inserting new
    (document, embedding) pairs into it.
    """

    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.persist_directory = persist_directory
        self.collection_name = collection_name
        self.client = None
        self.collection = None

        self._connect()

    def _connect(self):
        """Open (or create) the on-disk ChromaDB collection."""
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"},
        )

        print("Connected to collection:", self.collection_name)
        print("Documents currently in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        """Insert a batch of Document + embedding pairs into the collection.

        Every document needs a unique id, its text, its metadata, and its
        embedding — Chroma stores all four aligned by position.
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        ids, metadatas, texts, vectors = [], [], [], []

        for i, (document, embedding) in enumerate(zip(documents, embeddings)):
            ids.append(f"doc_{uuid.uuid4()}")

            metadata = dict(document.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(document.page_content)
            metadatas.append(metadata)

            texts.append(document.page_content)
            vectors.append(embedding.tolist())

        # Insert everything in a single call (not inside the loop) so we
        # don't accidentally add the same batch multiple times.
        self.collection.add(
            ids=ids,
            metadatas=metadatas,
            documents=texts,
            embeddings=vectors,
        )

        print("Documents added to vector store:", len(texts))
        print("Documents now in collection:", self.collection.count())

In [18]:
vector_store = VectorStoreManager()

Connected to collection: pdf_documents
Documents currently in collection: 0


In [19]:
# Full ingestion flow: chunks -> raw text -> embeddings -> saved in ChromaDB
chunk_texts = [chunk.page_content for chunk in document_chunks]

# If this fails, it means document_chunks is empty — scroll up and re-run the
# "raw_documents = load_all_pdfs()" and "document_chunks = split_into_chunks(...)"
# cells again (in that order) after confirming your PDF is in data/pdfs.
assert len(chunk_texts) > 0, (
    "chunk_texts is empty. Re-run the load_all_pdfs() and split_into_chunks() "
    "cells above (in order) before running this cell."
)

chunk_embeddings = embedding_manager.generate_embeddings(chunk_texts)

vector_store.add_documents(document_chunks, chunk_embeddings)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

Embeddings shape: (244, 384)
Documents added to vector store: 244
Documents now in collection: 244


## Retrieval Pipeline

Given a user's question, embed it with the same model, then ask ChromaDB for the
stored chunks whose vectors are closest to it — those are our most relevant pieces
of context.

In [20]:
class RAGRetriever:
    """Finds the top-k most semantically relevant chunks for a query."""

    def __init__(self, embedding_manager, vector_store):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        """Embed the query, search ChromaDB, and return the matching chunks.

        score_threshold filters out weak matches: similarity_score is
        computed as 1 - distance, so higher means more relevant.
        """
        query_vector = self.embedding_manager.generate_embeddings([query])[0]

        results = self.vector_store.collection.query(
            query_embeddings=[query_vector.tolist()],
            n_results=top_k,
        )

        matches = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            texts = results["documents"][0]
            distances = results["distances"][0]

            for rank, (doc_id, metadata, text, distance) in enumerate(
                zip(ids, metadatas, texts, distances), start=1
            ):
                similarity_score = 1 - distance

                if similarity_score >= score_threshold:
                    matches.append({
                        "id": doc_id,
                        "document": text,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": rank,
                    })

            print(f"Retrieved {len(matches)} matching chunks")
        else:
            print("No matching chunks found")

        return matches

In [21]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [22]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (1, 384)
Retrieved 0 matching chunks


[]

## Generation Pipeline

This is the "G" in RAG: take the chunks we just retrieved, drop them into a prompt as
context, and ask an LLM to answer using only that context. We use **Groq** here because
it offers a fast, free-tier API that's easy to set up for a student project — swap in
OpenAI, Gemini, or a local model if you'd prefer.

Get a free API key here: https://console.groq.com/keys

In [25]:
import os
from getpass import getpass

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

Enter your Groq API key: ··········


In [26]:
from langchain_groq import ChatGroq


class RAGPipeline:
    """Ties retrieval and generation together: retrieve context, then ask the LLM."""

    def __init__(self, retriever, model_name="llama-3.1-8b-instant", top_k=5):
        self.retriever = retriever
        self.top_k = top_k
        self.llm = ChatGroq(model=model_name, temperature=0)

    def _build_prompt(self, query, retrieved_chunks):
        """Assemble the retrieved chunks and the question into one prompt string."""
        context = "\n\n".join(
            f"[Source {i + 1}] {chunk['document']}"
            for i, chunk in enumerate(retrieved_chunks)
        )

        return f"""You are a helpful assistant answering questions using only the provided context.
If the answer is not contained in the context, say you don't know instead of guessing.

Context:
{context}

Question: {query}

Answer:"""

    def answer(self, query):
        """Run the full RAG flow for a single question and return the answer + sources."""
        retrieved_chunks = self.retriever.retrieve(query, top_k=self.top_k)

        if not retrieved_chunks:
            return {"answer": "No relevant context found in the documents.", "sources": []}

        prompt = self._build_prompt(query, retrieved_chunks)
        response = self.llm.invoke(prompt)

        sources = [chunk["metadata"].get("source", "unknown") for chunk in retrieved_chunks]

        return {
            "answer": response.content,
            "sources": sources,
            "retrieved_chunks": retrieved_chunks,
        }

In [27]:
rag_pipeline = RAGPipeline(rag_retriever)

result = rag_pipeline.answer("What is encoder decoder")

print("ANSWER:\n", result["answer"])
print("\nSOURCES:", result["sources"])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (1, 384)
Retrieved 0 matching chunks
ANSWER:
 No relevant context found in the documents.

SOURCES: []
